In [31]:
import pandas as pd
import numpy as np
import warnings
import argparse
import os
import sys

warnings.filterwarnings('ignore')

# parse command-line arguments (safe for notebooks)
parser = argparse.ArgumentParser(description='Credit risk grading analysis')
parser.add_argument('--export', action='store_true', dest='export_results',
                    help='Write summary tables to Excel and CSV')
parser.add_argument('--out', dest='export_path', help='Output file path (defaults to credit_risk_analysis.xlsx)')

# if running inside a Jupyter kernel, ignore extra args
if 'ipykernel' in sys.modules:
    args, _ = parser.parse_known_args()
else:
    args = parser.parse_args()

export_results = args.export_results
export_path = args.export_path

In [32]:
print("\n" + "="*80)
print("SECTION 1: LOADING AND INSPECTING DATA")
print("="*80)

# Load all CSV files
loan_core = pd.read_csv("loan_core.csv")
borrower = pd.read_csv("borrower_profile.csv")
credit_history = pd.read_csv("credit_history.csv", low_memory=False)
account_balances = pd.read_csv("account_balances.csv")
account_activity = pd.read_csv("account_activity.csv")
extra = pd.read_csv("extra_unassigned.csv", low_memory=False)

# Dataset overview
datasets = {
    'loan_core': loan_core,
    'borrower': borrower,
    'credit_history': credit_history,
    'account_balances': account_balances,
    'account_activity': account_activity,
    'extra': extra
}

print("\nDataset Row Counts and Alignment:")
print("-" * 80)
for name, df in datasets.items():
    print(f"{name:.<25} {df.shape[0]:>10,} rows | {df.shape[1]:>3} columns")

# Check for alignment on ID field
print("\nID Field Alignment Check:")
print("-" * 80)
print(f"All datasets have {loan_core.shape[0]:,} rows [OK] (Row-aligned)")

# Verify ID consistency where applicable
valid_ids_by_dataset = {
    'loan_core': loan_core['id'].notna().sum(),
    'borrower': borrower['id'].notna().sum(),
    'credit_history': credit_history['id'].notna().sum(),
    'account_balances': account_balances['id'].notna().sum(),
    'account_activity': account_activity['id'].notna().sum(),
    'extra': extra['id'].notna().sum() if 'id' in extra.columns else extra['member_id'].notna().sum()
}

print("\nValid ID counts:")
for ds, count in valid_ids_by_dataset.items():
    print(f"  {ds:.<20} {count:>10,} valid IDs")



SECTION 1: LOADING AND INSPECTING DATA

Dataset Row Counts and Alignment:
--------------------------------------------------------------------------------
loan_core................  2,260,668 rows |   6 columns
borrower.................  2,260,668 rows |   7 columns
credit_history...........  2,260,668 rows |   8 columns
account_balances.........  2,260,668 rows |  11 columns
account_activity.........  2,260,668 rows |  21 columns
extra....................  2,260,668 rows |  94 columns

ID Field Alignment Check:
--------------------------------------------------------------------------------
All datasets have 2,260,668 rows [OK] (Row-aligned)

Valid ID counts:
  loan_core...........          0 valid IDs
  borrower............          0 valid IDs
  credit_history......          0 valid IDs
  account_balances....          0 valid IDs
  account_activity....          0 valid IDs
  extra...............          0 valid IDs


In [33]:
print("\n" + "="*80)
print("SECTION 2: DATA PREPARATION AND DEFAULT FLAG DEFINITION")
print("="*80)

# Create binary default flag
# Default if: Charged Off, Does not meet credit policy – Charged Off, or Default
loan_core["default_flag"] = (
    loan_core["loan_status"].str.contains("Charged Off", case=False, na=False) |
    (loan_core["loan_status"] == "Default")
).astype(int)

print(f"\nDefault Flag Distribution:")
print("-" * 80)
default_counts = loan_core["default_flag"].value_counts()
default_pct = loan_core["default_flag"].value_counts(normalize=True) * 100

for status in [0, 1]:
    print(f"  Status {status}: {default_counts.get(status, 0):>10,} loans ({default_pct.get(status, 0):>6.2f}%)")

print(f"\nOverall Default Rate: {loan_core['default_flag'].mean():.4f} ({loan_core['default_flag'].mean()*100:.2f}%)")



SECTION 2: DATA PREPARATION AND DEFAULT FLAG DEFINITION

Default Flag Distribution:
--------------------------------------------------------------------------------
  Status 0:  1,998,221 loans ( 88.39%)
  Status 1:    262,447 loans ( 11.61%)

Overall Default Rate: 0.1161 (11.61%)


In [34]:
print("\n" + "="*80)
print("SECTION 3: DEFAULT RATE ANALYSIS BY KEY DIMENSIONS")
print("="*80)

# 3.1: Default Rate by Grade
print("\n\n3.1 DEFAULT RATE BY GRADE")
print("-" * 80)

default_by_grade = (
    loan_core.groupby("grade", observed=True)
    .agg({
        'default_flag': ['sum', 'count', 'mean']
    })
    .round(4)
)

default_by_grade.columns = ['Defaults', 'Total_Loans', 'Default_Rate']
default_by_grade = default_by_grade.sort_index()
default_by_grade['Default_Rate_%'] = (default_by_grade['Default_Rate'] * 100).round(2)

print(default_by_grade)
print(f"\nKey Insight: Default rates should generally increase from Grade A to G.")
print("Identify any inversions or unexpectedly high/low rates.")

#3.1 charting with plotly for better visualization

import plotly.express as px

# Recalculate clean plotting table
grade_plot = (
    loan_core
    .groupby("grade")["default_flag"]
    .mean()
    .reset_index()
)

grade_plot["Default Rate (%)"] = grade_plot["default_flag"] * 100

fig = px.bar(
    grade_plot,
    x="grade",
    y="Default Rate (%)",
    title="Default Rate by Grade",
    text="Default Rate (%)",
    color="Default Rate (%)",
    color_continuous_scale="Reds"
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(yaxis_title="Default Rate (%)")

fig.show()

# 3.2: Default Rate by Term
print("\n\n3.2 DEFAULT RATE BY LOAN TERM")
print("-" * 80)

default_by_term = (
    loan_core.groupby("term", observed=True)
    .agg({
        'default_flag': ['sum', 'count', 'mean']
    })
    .round(4)
)

default_by_term.columns = ['Defaults', 'Total_Loans', 'Default_Rate']
default_by_term['Default_Rate_%'] = (default_by_term['Default_Rate'] * 100).round(2)

print(default_by_term)
print(f"\nKey Insight: Longer-term loans may have higher default rates due to extended exposure.")

term_plot = (
    loan_core
    .groupby("term")["default_flag"]
    .mean()
    .reset_index()
)

term_plot["Default Rate (%)"] = term_plot["default_flag"] * 100

fig = px.bar(
    term_plot,
    x="term",
    y="Default Rate (%)",
    title="Default Rate by Loan Term",
    text="Default Rate (%)",
    color="Default Rate (%)",
    color_continuous_scale="Blues"
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.show()

# 3.3: Default Rate by Disbursement Method
print("\n\n3.3 DEFAULT RATE BY DISBURSEMENT METHOD")
print("-" * 80)

default_by_disburse = (
    loan_core.groupby("disbursement_method", observed=True)
    .agg({
        'default_flag': ['sum', 'count', 'mean']
    })
    .round(4)
)

default_by_disburse.columns = ['Defaults', 'Total_Loans', 'Default_Rate']
default_by_disburse['Default_Rate_%'] = (default_by_disburse['Default_Rate'] * 100).round(2)

print(default_by_disburse)
print(f"\nKey Insight: Look for material differences in default rates by disbursement method.")

# 3.4: Cross-tabulation of Grade and Term
print("\n\n3.4 DEFAULT RATE BY GRADE AND TERM")
print("-" * 80)

grade_term_crosstab = pd.crosstab(
    loan_core['grade'],
    loan_core['term'],
    values=loan_core['default_flag'],
    aggfunc='mean'
).round(4) * 100

print("\nDefault Rate (%) - Grade x Term Matrix:")
print(grade_term_crosstab.round(2))



SECTION 3: DEFAULT RATE ANALYSIS BY KEY DIMENSIONS


3.1 DEFAULT RATE BY GRADE
--------------------------------------------------------------------------------
       Defaults  Total_Loans  Default_Rate  Default_Rate_%
grade                                                     
A         13776       433027        0.0318            3.18
B         51168       663557        0.0771            7.71
C         83419       650053        0.1283           12.83
D         59646       324424        0.1839           18.39
E         35526       135639        0.2619           26.19
F         14358        41800        0.3435           34.35
G          4554        12168        0.3743           37.43

Key Insight: Default rates should generally increase from Grade A to G.
Identify any inversions or unexpectedly high/low rates.




3.2 DEFAULT RATE BY LOAN TERM
--------------------------------------------------------------------------------
           Defaults  Total_Loans  Default_Rate  Default_Rate_%
term                                                          
36 months    159818      1609754        0.0993            9.93
60 months    102629       650914        0.1577           15.77

Key Insight: Longer-term loans may have higher default rates due to extended exposure.




3.3 DEFAULT RATE BY DISBURSEMENT METHOD
--------------------------------------------------------------------------------
                     Defaults  Total_Loans  Default_Rate  Default_Rate_%
disbursement_method                                                     
Cash                   261200      2182546        0.1197           11.97
DirectPay                1247        78122        0.0160            1.60

Key Insight: Look for material differences in default rates by disbursement method.


3.4 DEFAULT RATE BY GRADE AND TERM
--------------------------------------------------------------------------------

Default Rate (%) - Grade x Term Matrix:
term   36 months  60 months
grade                      
A           3.21       2.64
B           8.01       6.47
C          13.17      12.21
D          18.06      18.82
E          22.99      28.11
F          32.30      34.89
G          36.49      37.62


In [35]:
full_data["income_quintile"] = pd.qcut(
    full_data["annual_inc"],
    5,
    labels=["Q1","Q2","Q3","Q4","Q5"],
    duplicates="drop"
)

In [36]:
full_data["income_quintile"].value_counts()

income_quintile
Q3    477819
Q1    457807
Q2    457710
Q5    447153
Q4    420175
Name: count, dtype: int64

In [53]:
print("\n" + "="*80)
print("SECTION 4: BORROWER PROFILE ANALYSIS")
print("="*80)

# The extra dataset appears to contain critical info: let's use it as base
# Merge loan_core with other datasets
# Since all datasets are row-aligned and IDs are all NaN, use index-based concatenation

# Reset indices and concatenate horizontally (by column)
loan_core_reset = loan_core.reset_index(drop=True)
extra_reset = extra.reset_index(drop=True)
borrower_reset = borrower.reset_index(drop=True)
credit_history_reset = credit_history.reset_index(drop=True)
account_balances_reset = account_balances.reset_index(drop=True)
account_activity_reset = account_activity.reset_index(drop=True)

# Combine all datasets using pd.concat on axis=1 (columns)
full_data = pd.concat([
    loan_core_reset,
    extra_reset[[col for col in extra_reset.columns if col not in loan_core_reset.columns]].iloc[:, :-1],
    borrower_reset[[col for col in borrower_reset.columns if col not in loan_core_reset.columns]],
    credit_history_reset[[col for col in credit_history_reset.columns if col not in loan_core_reset.columns]],
    account_balances_reset[[col for col in account_balances_reset.columns if col not in loan_core_reset.columns]],
    account_activity_reset[[col for col in account_activity_reset.columns if col not in loan_core_reset.columns]]
], axis=1)

print(f"\nFull Dataset Shape: {full_data.shape}")
print(f"Data Completeness: {(1 - full_data.isna().sum().sum() / (full_data.shape[0] * full_data.shape[1])) * 100:.1f}%")

# 4.1: Income Analysis
print("\n\n4.1 INCOME DISTRIBUTION BY GRADE")
print("-" * 80)

if 'annual_inc' in full_data.columns:
    income_by_grade = (
        full_data.dropna(subset=['annual_inc'])
        .groupby('grade')['annual_inc']
        .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        .round(2)
    )
    print(income_by_grade)
    print(f"\nKey Insight: Check if grade assignment aligns with income profiles.")
    print(f"(Higher grades should generally reflect higher/more stable income)")
    
    import plotly.express as px

income_grade = full_data.groupby("grade")["annual_inc"].mean().reset_index()

fig = px.bar(
    income_grade,
    x="grade",
    y="annual_inc",
    title="Average Income by Grade",
    color="annual_inc",
    color_continuous_scale="Viridis"
)

fig.update_layout(yaxis_title="Average Annual Income")
fig.show()

# 4.2: Default Rate by Income Bands
print("\n\n4.2 DEFAULT RATE BY INCOME BANDS")
print("-" * 80)

if 'annual_inc' in full_data.columns:
    full_data_clean = full_data.dropna(subset=['annual_inc'])
    
    # Create income quintiles
    full_data_clean['income_quintile'] = pd.qcut(
        full_data_clean['annual_inc'],
        q=5,
        labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (Highest)'],
        duplicates='drop'
    )
    
    default_by_income = (
        full_data_clean.groupby('income_quintile')
        .agg({
            'default_flag': ['sum', 'count', 'mean']
        })
        .round(4)
    )
    
    default_by_income.columns = ['Defaults', 'Total_Loans', 'Default_Rate']
    default_by_income['Default_Rate_%'] = (default_by_income['Default_Rate'] * 100).round(2)
    
    print(default_by_income)
    print(f"\nKey Insight: Higher income should correlate with lower default rates.")
    print(f"Anomaly Check: High-income borrowers defaulting at high rates = risk signal.")
    
   
# 4.4: Grade vs Income Alignment Check
print("\n\n4.4 GRADE vs INCOME ALIGNMENT - POTENTIAL MISMATCHES")
print("-" * 80)

if 'annual_inc' in full_data.columns:
    # Create income categories
    full_data_check = full_data.dropna(subset=['annual_inc'])
    full_data_check['income_category'] = pd.cut(
        full_data_check['annual_inc'],
        bins=[0, 30000, 60000, 100000, np.inf],
        labels=['Low (<30K)', 'Medium (30-60K)', 'High (60-100K)', 'Very High (>100K)']
    )
    
    grade_income_mismatch = pd.crosstab(
        full_data_check['grade'],
        full_data_check['income_category'],
        margins=True
    )
    
    print("\nCross-tabulation: Grade vs Income Category")
    print(grade_income_mismatch)
    
    # Flag potential misclassifications
    print("\n\nPotential Grade-Income Mismatches:")
    print("-" * 50)
    
    # E.g., high-grade (A-C) borrowers with low income
    low_income_high_grade = full_data_check[
        (full_data_check['grade'].isin(['A', 'B', 'C'])) &
        (full_data_check['income_category'] == 'Low (<30K)')
    ]
    
    print(f"[WARNING] Grade A-C borrowers with LOW income (<30K): {len(low_income_high_grade)} loans ({(len(low_income_high_grade)/len(full_data_check)*100):.1f}%)")
    print(f"  Default rate in this group: {low_income_high_grade['default_flag'].mean()*100:.2f}%")
    
    # E.g., low-grade (F-G) borrowers with high income
    high_income_low_grade = full_data_check[
        (full_data_check['grade'].isin(['F', 'G'])) &
        (full_data_check['income_category'] == 'Very High (>100K)')
    ]
    
    print(f"[WARNING] Grade F-G borrowers with HIGH income (>100K): {len(high_income_low_grade):,} loans ({len(high_income_low_grade)/len(full_data_check)*100:.1f}%)")
    print(f"  Default rate in this group: {high_income_low_grade['default_flag'].mean()*100:.2f}%")


SECTION 4: BORROWER PROFILE ANALYSIS

Full Dataset Shape: (2260668, 143)
Data Completeness: 68.4%


4.1 INCOME DISTRIBUTION BY GRADE
--------------------------------------------------------------------------------
        count      mean   median        std  min          max
grade                                                        
A      433023  89902.40  75000.0   90257.59  0.0    9573072.0
B      663557  78813.57  65000.0  171206.83  0.0  110000000.0
C      650053  74370.29  63000.0   72636.16  0.0    9930475.0
D      324424  71030.18  60000.0   68770.88  0.0   10999200.0
E      135639  71855.51  60000.0   63392.87  0.0    8500000.0
F       41800  72657.78  63000.0   49037.10  0.0    2500000.0
G       12168  75241.69  65000.0   51027.56  0.0     980000.0

Key Insight: Check if grade assignment aligns with income profiles.
(Higher grades should generally reflect higher/more stable income)




4.2 DEFAULT RATE BY INCOME BANDS
--------------------------------------------------------------------------------
                 Defaults  Total_Loans  Default_Rate  Default_Rate_%
income_quintile                                                     
Q1 (Lowest)         63248       457807        0.1382           13.82
Q2                  59334       457710        0.1296           12.96
Q3                  57013       477819        0.1193           11.93
Q4                  44577       420175        0.1061           10.61
Q5 (Highest)        38275       447153        0.0856            8.56

Key Insight: Higher income should correlate with lower default rates.
Anomaly Check: High-income borrowers defaulting at high rates = risk signal.


4.4 GRADE vs INCOME ALIGNMENT - POTENTIAL MISMATCHES
--------------------------------------------------------------------------------

Cross-tabulation: Grade vs Income Category
income_category  Low (<30K)  Medium (30-60K)  High (60-100K)  \
grade    

In [ ]:
print(credit_history.columns)


Index(['id', 'sec_app_earliest_cr_line', 'mths_since_last_major_derog',
       'chargeoff_within_12_mths', 'collections_12_mths_ex_med',
       'pub_rec_bankruptcies', 'tax_liens', 'pct_tl_nvr_dlq'],
      dtype='str')


In [54]:
print("\n" + "="*80)
print("SECTION 5: CREDIT HISTORY ALIGNMENT WITH GRADE")
print("="*80)

# Key credit history metrics
credit_metrics = ['mths_since_last_major_derog', 'chargeoff_within_12_mths', 
                  'pub_rec_bankruptcies', 'collections_12_mths_ex_med', 'pct_tl_nvr_dlq']

print("\n\n5.1 CREDIT METRIC DISTRIBUTION BY GRADE")
print("-" * 80)

for metric in credit_metrics:
    if metric in full_data.columns:
        print(f"\n{metric.upper()}:")
        metric_by_grade = (
            full_data.dropna(subset=[metric])
            .groupby('grade')[metric]
            .agg(['count', 'mean', 'median', 'std'])
            .round(3)
        )
        print(metric_by_grade)
        
        
# 5.2: Credit Quality Alignment
print("\n\n5.2 INDICATORS OF WEAK CREDIT HISTORY BY GRADE")
print("-" * 80)

if 'chargeoff_within_12_mths' in full_data.columns:
    chargeoff_by_grade = (
        full_data.groupby('grade')['chargeoff_within_12_mths']
        .agg(['sum', 'mean'])
        .round(4)
    )
    chargeoff_by_grade.columns = ['Count', 'Proportion']
    print("\nBorrowers with recent chargeoffs by grade:")
    print(chargeoff_by_grade)
else:
    print("\n[INFO] chargeoff_within_12_mths not available")

if 'pub_rec_bankruptcies' in full_data.columns:
    bankruptcy_by_grade = (
        full_data[full_data['pub_rec_bankruptcies'] > 0]
        .groupby('grade')['pub_rec_bankruptcies']
        .agg(['sum', 'count', 'mean'])
        .round(3)
    )
    bankruptcy_by_grade.columns = ['Total_Bankruptcies', 'Borrowers_With_Bankruptcy', 'Avg_Per_Borrower']
    print("\nBankruptcy history by grade:")
    print(bankruptcy_by_grade)



SECTION 5: CREDIT HISTORY ALIGNMENT WITH GRADE


5.1 CREDIT METRIC DISTRIBUTION BY GRADE
--------------------------------------------------------------------------------

MTHS_SINCE_LAST_MAJOR_DEROG:
        count    mean  median     std
grade                                
A       70653  46.767    47.0  20.948
B      171988  44.168    44.0  21.613
C      186218  43.744    43.0  21.586
D       96351  43.392    43.0  21.557
E       39905  43.493    43.0  21.614
F       12088  43.830    43.0  21.364
G        3572  43.854    44.0  21.548

CHARGEOFF_WITHIN_12_MTHS:
        count   mean  median    std
grade                              
A      432992  0.004     0.0  0.074
B      663528  0.009     0.0  0.107
C      650026  0.010     0.0  0.113
D      324407  0.010     0.0  0.114
E      135621  0.010     0.0  0.116
F       41788  0.009     0.0  0.105
G       12161  0.008     0.0  0.092

PUB_REC_BANKRUPTCIES:
        count   mean  median    std
grade                              
A      4328

In [ ]:
print("\n" + "="*80)
print("SECTION 6: DRIFT ANALYSIS - TEMPORAL STABILITY")
print("="*80)

# Check for time variables
time_columns = [col for col in full_data.columns if 'date' in col.lower() or 'd' in col.lower()]
print(f"\nPotential date columns found: {time_columns[:10]}")

# If issue_d exists, use it for trend analysis
if 'issue_d' in full_data.columns:
    print("\n\n6.1 DEFAULT RATE BY GRADE OVER TIME")
    print("-" * 80)
    
    # Parse issue date
    full_data['issue_date'] = pd.to_datetime(full_data['issue_d'], errors='coerce')
    full_data['issue_year'] = full_data['issue_date'].dt.year
    
    if full_data['issue_year'].notna().sum() > 0:
        # Default rate by year and grade
        trend_data = (
            full_data.dropna(subset=['issue_year'])
            .groupby(['issue_year', 'grade'])['default_flag']
            .agg(['mean', 'count'])
            .round(4)
        )
        
        trend_data.columns = ['Default_Rate', 'Loan_Count']
        trend_data.index.names = ['Year', 'Grade']
        
        print(trend_data)
        
        # Identify grade compression (all grades converging to same rate)
        print("\n\n6.2 GRADE COMPRESSION CHECK (Standard Deviation of Default Rates by Year)")
        print("-" * 80)
        
        compression_check = (
            full_data.dropna(subset=['issue_year'])
            .groupby('issue_year')
            .apply(lambda g: g.groupby('grade')['default_flag'].mean().std())
            .round(4)
        )
        
        print(compression_check)
        
        min_std = compression_check.min()
        max_std = compression_check.max()
        pct_change = ((min_std - max_std) / max_std) * 100
        
        print(f"\nGrade Separation (Std Dev of rates across grades):")
        print(f"  Highest: {max_std:.4f} | Lowest: {min_std:.4f}")
        print(f"  Change: {pct_change:.1f}%")
        
        if pct_change < -20:
            print(f"  [WARNING] Grade compression detected! Grades becoming indistinguishable.")
        else:
            print(f"  [OK] Grade separation remains stable.")
else:
    print("\nNo 'issue_d' date column found. Time-based drift analysis not available.")



SECTION 6: DRIFT ANALYSIS - TEMPORAL STABILITY

Potential date columns found: ['id', 'grade', 'disbursement_method', 'default_flag', 'member_id', 'funded_amnt', 'funded_amnt_inv', 'sub_grade', 'issue_d', 'addr_state']


6.1 DEFAULT RATE BY GRADE OVER TIME
--------------------------------------------------------------------------------
            Default_Rate  Loan_Count
Year Grade                          
2007 A            0.0513          78
     B            0.1633          98
     C            0.2411         141
     D            0.3131          99
     E            0.3100         100
...                  ...         ...
2018 C            0.0168      126850
     D            0.0286       69046
     E            0.0373       18958
     F            0.0551        3175
     G            0.0686         671

[84 rows x 2 columns]


6.2 GRADE COMPRESSION CHECK (Standard Deviation of Default Rates by Year)
--------------------------------------------------------------------------------
i

In [ ]:
print("\n" + "="*80)
print("SECTION 7: ANOMALY DETECTION AND RISK SIGNALS")
print("="*80)

# 7.1: High-Income Defaulters
print("\n\n7.1 HIGH-INCOME BORROWERS WITH DEFAULT (Anomaly)")
print("-" * 80)

if 'annual_inc' in full_data.columns:
    high_income_thresh = full_data['annual_inc'].quantile(0.75)
    high_income_defaults = full_data[
        (full_data['annual_inc'] >= high_income_thresh) &
        (full_data['default_flag'] == 1)
    ]
    
    print(f"High-income threshold (75th percentile): ${high_income_thresh:,.0f}")
    print(f"High-income defaulters: {len(high_income_defaults):,} ({len(high_income_defaults)/full_data['default_flag'].sum()*100:.1f}% of all defaults)")
    
    if len(high_income_defaults) > 0:
        print(f"\nTop features of high-income defaulters:")
        print(f"  Avg income: ${high_income_defaults['annual_inc'].mean():,.0f}")
        print(f"  Grade distribution:")
        print(high_income_defaults['grade'].value_counts())
        print(f"  Avg loan amount: ${high_income_defaults['loan_amnt'].mean():,.0f}")
        
        print(f"\n[ALERT] Investigation Needed: Why do wealthy borrowers default?")
        print(f"   - Potentially fraudulent applications?")
        print(f"   - Overstated income in applications?")
        print(f"   - Loan size/amount mismatch?")

# 7.2: Grade Concentration Anomalies
print("\n\n7.2 LOW-RISK GRADE WITH ELEVATED DEFAULT RATE (Anomaly)")
print("-" * 80)

overall_default_rate = loan_core['default_flag'].mean()
grade_stats = default_by_grade.copy()
grade_high_variance = grade_stats[grade_stats['Default_Rate_%'] > overall_default_rate * 150]

if len(grade_high_variance) > 0:
    print(f"Grades with default rate >150% of platform average:")
    print(grade_high_variance)
    print(f"\nRisk Signals:")
    for grade in grade_high_variance.index:
        underpriced = (grade_high_variance.loc[grade, 'Default_Rate_%'] - overall_default_rate * 100) / overall_default_rate * 100
        print(f"  Grade {grade}: {underpriced:.0f}% above average. UNDERPRICED RISK")
else:
    print("[OK] No significant anomalies detected in grade-based default rates.")

# 7.3: Similar Borrowers, Different Outcomes
print("\n\n7.3 SIMILAR BORROWERS WITH DIVERGENT OUTCOMES")
print("-" * 80)

# Compare borrowers with similar credit profiles (income, credit score proxies)
if 'annual_inc' in full_data.columns and 'pub_rec_bankruptcies' in full_data.columns:
    income_band = full_data[full_data['annual_inc'].between(50000, 60000)]
    
    if len(income_band) > 100:
        print(f"\nAnalyzing {len(income_band):,} borrowers in $50-60K income band:")
        
        # Group by bankruptcy history (proxy for credit quality)
        by_bankruptcy = income_band.groupby('pub_rec_bankruptcies')['default_flag'].agg(['count', 'mean', 'sum'])
        by_bankruptcy.columns = ['Count', 'Default_Rate', 'Defaults']
        
        print(by_bankruptcy)
        print(f"\nKey Finding: Similar-income borrowers with/without bankruptcy history")
        print(f"  show divergent default rates. This is expected and validates grade logic.")

# 7.4: Loan Amount vs Default
print("\n\n7.4 LOAN AMOUNT ANALYSIS")
print("-" * 80)

if 'loan_amnt' in full_data.columns:
    # Create loan amount quintiles
    full_data['loan_amt_bucket'] = pd.qcut(
        full_data['loan_amnt'].dropna(),
        q=5,
        labels=['Very Small', 'Small', 'Medium', 'Large', 'Very Large'],
        duplicates='drop'
    )
    
    # For those with NA: assign a bucket
    full_data['loan_amt_bucket'] = full_data['loan_amt_bucket'].astype(str)
    
    loan_size_default = (
        full_data[full_data['loan_amt_bucket'] != 'nan']
        .groupby('loan_amt_bucket')
        .agg({'default_flag': ['count', 'sum', 'mean']})
        .round(4)
    )
    
    loan_size_default.columns = ['Count', 'Defaults', 'Default_Rate']
    loan_size_default['Default_Rate_%'] = (loan_size_default['Default_Rate'] * 100).round(2)
    
    print(loan_size_default)



SECTION 7: ANOMALY DETECTION AND RISK SIGNALS


7.1 HIGH-INCOME BORROWERS WITH DEFAULT (Anomaly)
--------------------------------------------------------------------------------
High-income threshold (75th percentile): $93,000
High-income defaulters: 50,650 (19.3% of all defaults)

Top features of high-income defaulters:
  Avg income: $137,048
  Grade distribution:
grade
C    15768
B    10794
D    10268
E     6432
A     3801
F     2661
G      926
Name: count, dtype: int64
  Avg loan amount: $22,497

[ALERT] Investigation Needed: Why do wealthy borrowers default?
   - Potentially fraudulent applications?
   - Overstated income in applications?
   - Loan size/amount mismatch?


7.2 LOW-RISK GRADE WITH ELEVATED DEFAULT RATE (Anomaly)
--------------------------------------------------------------------------------
Grades with default rate >150% of platform average:
       Defaults  Total_Loans  Default_Rate  Default_Rate_%
grade                                                     
D      

In [ ]:
print("\n" + "="*80)
print("SECTION 8: EXECUTIVE SUMMARY AND KEY FINDINGS")
print("="*80)

print("""
================================================================================
                     CREDIT RISK ASSESSMENT SUMMARY
================================================================================

KEY METRICS:
""")

print(f"""
  - Total Loans Analyzed:              {loan_core.shape[0]:>15,}
  - Overall Default Rate:              {loan_core['default_flag'].mean()*100:>14.2f}%
  - Total Defaults:                    {loan_core['default_flag'].sum():>15,}
  - Active/Current Loans:              {(loan_core['loan_status']=='Current').sum():>15,}

GRADE DISTRIBUTION & RISK:
""")

for grade in sorted(loan_core['grade'].unique()):
    grade_data = loan_core[loan_core['grade'] == grade]
    default_rate = grade_data['default_flag'].mean()
    count = len(grade_data)
    risk_bars = '*' * int(default_rate*20)
    print(f"  * Grade {grade:1s}:  {count:>10,} loans | Default Rate: {default_rate*100:>6.2f}% | Risk Level: {risk_bars}")

print(f"""

CRITICAL FINDINGS:
""")

# Finding 1: Grade Predictiveness
if default_by_grade['Default_Rate'].iloc[0] < (overall_default_rate * 0.7) and \
   default_by_grade['Default_Rate'].iloc[-1] > (overall_default_rate * 1.5):
    print(f"  [OK] FINDING 1: Grade System is PREDICTIVE")
    print(f"    - Lowest risk grade has {default_by_grade['Default_Rate'].iloc[0]*100:.2f}% default rate")
    print(f"    - Highest risk grade has {default_by_grade['Default_Rate'].iloc[-1]*100:.2f}% default rate")
    print(f"    - Spread indicates meaningful risk differentiation\n")
else:
    print(f"  [WARNING] FINDING 1: Grade System may LACK SEPARATION")
    print(f"    - Grades not showing expected risk progression")
    print(f"    - Recommend review of grading methodology\n")

# Finding 2: Income-Grade Alignment
if 'annual_inc' in full_data.columns:
    income_grade_corr = full_data.dropna(subset=['annual_inc', 'grade']).groupby('grade')['annual_inc'].mean()
    # Check if income increases with grade (A < B < C ... < G)
    sorted_grades = sorted(income_grade_corr.index)
    values_ascending = income_grade_corr[sorted_grades].values
    expected_order = all(values_ascending[i] <= values_ascending[i+1] for i in range(len(values_ascending)-1))
    
    if expected_order:
        print(f"  [OK] FINDING 2: Income ALIGNS with Grade (higher grades = higher income)\n")
    else:
        print(f"  [WARNING] FINDING 2: Income MISALIGNED with Grade")
        print(f"    - Some grades receive higher income than expected")
        print(f"    - Possible grading criteria inconsistency\n")

# Finding 3: Stability
if 'issue_year' in full_data.columns and full_data['issue_year'].notna().sum() > 0:
    first_year_std = full_data[full_data['issue_year'] == full_data['issue_year'].min()].groupby('grade')['default_flag'].mean().std()
    last_year_std = full_data[full_data['issue_year'] == full_data['issue_year'].max()].groupby('grade')['default_flag'].mean().std()
    
    if abs(last_year_std - first_year_std) / first_year_std < 0.20:
        print(f"  [OK] FINDING 3: Grade System is STABLE over time\n")
    else:
        print(f"  [WARNING] FINDING 3: Grade System shows DRIFT over time")
        print(f"    - Grade separation declining (compression detected)")
        print(f"    - Recommend recalibration\n")

print(f"""
ANOMALIES AND RISK SIGNALS:
""")

if 'annual_inc' in full_data.columns:
    high_income_low_grade_count = len(full_data[
        (full_data['annual_inc'] >= full_data['annual_inc'].quantile(0.75)) &
        (full_data['grade'].isin(['F', 'G']))
    ])
    
    if high_income_low_grade_count > len(full_data) * 0.001:
        print(f"  [WARNING] {high_income_low_grade_count:,} high-income borrowers assigned low grades (F-G)")
        print(f"    - Potential underutilized borrower quality\n")

print(f"""
RECOMMENDATIONS:
  1. Review grade assignment methodology for consistency
  2. Validate income and employment data quality
  3. Assess whether interest rates appropriately reflect grade risk
  4. If drift detected: Trigger model retraining and recalibration
  5. Investigate high-income defaulters for fraud indicators
  6. Monitor credit quality metrics going forward
  7. Consider interim pricing adjustments for mispriced grades
  
  
  
# """)

# print("\n" + "="*80)
# print("Analysis Complete")
# print("="*80)



SECTION 8: EXECUTIVE SUMMARY AND KEY FINDINGS

                     CREDIT RISK ASSESSMENT SUMMARY

KEY METRICS:


  - Total Loans Analyzed:                    2,260,668
  - Overall Default Rate:                       11.61%
  - Total Defaults:                            262,447
  - Active/Current Loans:                      919,695

GRADE DISTRIBUTION & RISK:

  * Grade A:     433,027 loans | Default Rate:   3.18% | Risk Level: 
  * Grade B:     663,557 loans | Default Rate:   7.71% | Risk Level: *
  * Grade C:     650,053 loans | Default Rate:  12.83% | Risk Level: **
  * Grade D:     324,424 loans | Default Rate:  18.39% | Risk Level: ***
  * Grade E:     135,639 loans | Default Rate:  26.19% | Risk Level: *****
  * Grade F:      41,800 loans | Default Rate:  34.35% | Risk Level: ******
  * Grade G:      12,168 loans | Default Rate:  37.43% | Risk Level: *******


CRITICAL FINDINGS:

  [OK] FINDING 1: Grade System is PREDICTIVE
    - Lowest risk grade has 3.18% default rate
    - H

In [ ]:
# optional export path provided via argument
if export_results:
    out_file = export_path or 'credit_risk_analysis.xlsx'
    print(f"\nExporting summary tables to {out_file}...")
    with pd.ExcelWriter(out_file, engine='openpyxl') as writer:
        default_by_grade.to_excel(writer, sheet_name='Default_by_Grade')
        default_by_term.to_excel(writer, sheet_name='Default_by_Term')
        default_by_disburse.to_excel(writer, sheet_name='Default_by_Disburse')
        
        if 'income_quintile' in full_data.columns:
            default_by_income.to_excel(writer, sheet_name='Default_by_Income')
        
        if 'loan_amt_bucket' in full_data.columns:
            loan_size_default.to_excel(writer, sheet_name='Default_by_LoanSize')

    # also export raw combined dataset if desired
    raw_csv = out_file.replace('.xlsx', '_full.csv')
    full_data.to_csv(raw_csv, index=False)
    print(f"[OK] Full combined data exported to {raw_csv}")

# """

